In [1]:
import torch
import torch.nn as nn
import timm

class PaperHybridViT(nn.Module):
    """
    
    "Multi-label classification of chest X-ray images with pre-trained vision Transformer model"
    """
    def __init__(self, num_classes=14, in_channels=1):
        super(PaperHybridViT, self).__init__()
        
        # ---------------------------------------------------------------------
        # BƯỚC 1 & 2 CỦA BÀI BÁO (Mục 2.4): 
        # Chuyển đổi kênh màu (Từ ảnh Xám 1 kênh sang 3 kênh RGB)
        # Bài báo ghi: "sử dụng 1x1 conv layer để điều chỉnh số kênh ảnh X-quang"
        # ---------------------------------------------------------------------
        self.in_channels = in_channels
        if self.in_channels == 1:
            self.channel_converter = nn.Conv2d(1, 3, kernel_size=1, stride=1, padding=0)
        else:
            self.channel_converter = nn.Identity()

        # ---------------------------------------------------------------------
        # BƯỚC 3 CỦA BÀI BÁO: Backbone CNN + Transformer (Pre-trained)
        # Bài báo sử dụng mô hình ViT lai CNN (Hybrid).
        # Cụ thể Bảng 1 mô tả cấu trúc CNN lấy từ ResNet50 (tới Block 3).
        # Trong thư viện `timm`, kiến trúc này chính là: 'vit_base_resnet50_224_in21k'
        # Nó đã được Pre-train trên tập khổng lồ (giống JFT-300M/ImageNet như bài báo mô tả)
        # ---------------------------------------------------------------------
        self.backbone = timm.create_model('vit_base_resnet50_224_in21k', pretrained=True)
        
        # ---------------------------------------------------------------------
        # BƯỚC 4 CỦA BÀI BÁO: Phân loại Đa nhãn (Classifier)
        # Bài báo ghi: "đầu ra số node từ 1000 điều chỉnh về số nhãn của X-quang"
        # ---------------------------------------------------------------------
        # SỬA LỖI TIMM: Dùng .num_features thay vì .head.in_features 
        # do in21k model có thể đã bị timm mặc định thay head bằng Identity()
        in_features = self.backbone.num_features
        self.backbone.head = nn.Linear(in_features, num_classes)
        
        # Lưu ý: Bài báo nhắc đến "Sigmoid" ở đầu ra. 
        # Trong PyTorch, ta KHÔNG đặt Sigmoid ở đây, mà dùng hàm loss BCEWithLogitsLoss lúc train,
        # vì BCEWithLogitsLoss đã tích hợp sẵn Sigmoid bên trong để tính toán ổn định hơn.

    def forward(self, x):
        # 1. Chuyển từ 1 kênh sang 3 kênh (Nếu dùng ảnh xám)
        x = self.channel_converter(x)
        
        # 2. Đi qua mạng CNN (ResNet stem) + Transformer
        logits = self.backbone(x)
        
        return logits

Đang tải kiến trúc Hybrid ViT-ResNet50...


/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name vit_base_resnet50_224_in21k to current vit_base_r50_s16_224.orig_in21k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/392M [00:00<?, ?B/s]


Kích thước Input: torch.Size([8, 1, 224, 224]) (Ảnh X-quang xám)
Kích thước Output: torch.Size([8, 14])

Thành công! Mô hình đã ép kiểu 1 kênh thành 3 kênh và xuất ra 14 dự đoán y như mô tả trong bài báo gốc Trung Quốc.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score
import numpy as np
from tqdm import tqdm
import warnings
import os
from PIL import Image
import torchvision.transforms as transforms
import pandas as pd
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. TEMPLATE DATASET CHEXPERT (Bạn ghép code đọc CSV của bạn vào đây)
# ---------------------------------------------------------
class CheXpertDataset(Dataset):
    def __init__(self, df, transform=None):
        """
        df: DataFrame chứa đường dẫn ảnh và 14 cột nhãn (đã xử lý U-Zeros)
        """
        self.df = df
        self.transform = transform
        self.labels = self.df.iloc[:, -14:].values.astype(np.float32)
        self.image_paths = self.df['Path'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Đọc ảnh X-quang bằng PIL (chuyển về Grayscale 1 kênh 'L')
        img_path = self.image_paths[idx]
        try:
            img = Image.open(img_path).convert('L')
        except Exception:
            # Nếu đường dẫn lỗi, tạo ảnh đen tạm thời để không bị crash training
            img = Image.new('L', (224, 224))
            
        # Áp dụng các phép biến đổi (Resize, Normalize)
        if self.transform:
            img = self.transform(img)
            
        label = torch.tensor(self.labels[idx])
        return img, label

# ---------------------------------------------------------
# 2. HÀM HUẤN LUYỆN 1 EPOCH (TỐI ƯU CHO KAGGLE)
# ---------------------------------------------------------
def train_epoch(model, loader, criterion, optimizer, scaler, device, accumulation_steps=4):
    model.train()
    running_loss = 0.0
    
    # Thanh tiến trình (tqdm) để theo dõi trên Kaggle
    pbar = tqdm(loader, desc="Training")
    
    optimizer.zero_grad()
    
    for i, (images, labels) in enumerate(pbar):
        images, labels = images.to(device), labels.to(device)
        
        
        # Ép kiểu tính toán về Float16 
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
            # Khắc phục khi dùng Gradient Accumulation
            loss = loss / accumulation_steps 
            
        #  Backward với GradScaler 
        scaler.scale(loss).backward()
        
        # VŨ KHÍ 3: Gradient Accumulation
        
        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        running_loss += loss.item() * accumulation_steps
        pbar.set_postfix({'loss': f"{running_loss/(i+1):.4f}"})
        
    return running_loss / len(loader)

# ---------------------------------------------------------
# 3. HÀM ĐÁNH GIÁ (VALIDATION) VÀ TÍNH AUROC
# ---------------------------------------------------------
def evaluate(model, loader, criterion, device):
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            # Dùng Sigmoid để chuyển logits thành xác suất từ 0 đến 1
            preds = torch.sigmoid(outputs).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(labels.cpu().numpy())
            
    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    
    # Tính MACRO AUROC
    try:
        auroc = roc_auc_score(all_targets, all_preds, average='macro')
    except ValueError:
        auroc = 0.0 # Bắt lỗi nếu batch quá nhỏ chỉ có 1 class
        
    return val_loss / len(loader), auroc

# ---------------------------------------------------------
# 4. KỊCH BẢN CHÍNH (MAIN EXECUTION)
# ---------------------------------------------------------
if __name__ == "__main__":
    # Khởi tạo thiết bị
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Đang chạy trên thiết bị: {device}")
    
    # ==========================================
    # BƯỚC 1: ĐỌC VÀ TIỀN XỬ LÝ DỮ LIỆU CHEXPERT
    # ==========================================
    # ĐƯỜNG DẪN TỚI FILE CSV TRÊN KAGGLE (Bạn cần sửa đường dẫn này)

    csv_path = '/kaggle/input/datasets/willarevalo/chexpert-v10-small/CheXpert-v1.0-small/train.csv' 
    
    print("Đang nạp file dữ liệu CSV...")
    try:
        df = pd.read_csv(csv_path)
        # THỰC THI CHIẾN LƯỢC U-ZEROS (Như đã báo cáo trong Khóa luận)
        df = df.fillna(0)            # Blank (NaN) -> 0
        df = df.replace(-1.0, 0.0)   # Uncertain (-1) -> 0
        
        # MẸO KAGGLE: Nếu đường dẫn trong CSV chưa có thư mục gốc, hãy cộng chuỗi vào
        df['Path'] = '/kaggle/input/datasets/willarevalo/chexpert-v10-small/' + df['Path']
        
        # Chia tập Train / Val (Lấy 90% train, 10% val)
        train_df = df.sample(frac=0.9, random_state=42)
        val_df = df.drop(train_df.index)
    except FileNotFoundError:
        print("CẢNH BÁO: Chưa tìm thấy file CSV. Đang tạo dữ liệu giả lập để test...")
        dummy_data = {'Path': ['dummy.jpg']*100}
        for i in range(14):
            dummy_data[f'Disease_{i}'] = np.random.randint(0, 2, 100)
        df = pd.DataFrame(dummy_data)
        train_df = df.iloc[:80]
        val_df = df.iloc[80:]

    # Data Augmentation (Tăng cường dữ liệu) theo bài báo
    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]) # Chuẩn hóa ảnh 1 kênh (Grayscale)
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    
    # Tạo Dataset và DataLoader
    train_dataset = CheXpertDataset(train_df, transform=train_transform)
    val_dataset = CheXpertDataset(val_df, transform=val_transform)
    
    # Tự động điều chỉnh num_workers để Kaggle load data song song nhanh hơn
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

    # ==========================================
    # BƯỚC 2: KHỞI TẠO MÔ HÌNH HYBRID VIT
    # ==========================================
    print("Đang khởi tạo mô hình Hybrid ViT...")
    
    model = PaperHybridViT(num_classes=14, in_channels=1).to(device)
    
   
    
    # Cấu hình Hyperparameters (Theo đúng bài báo)
    BATCH_SIZE = 8          # Batch size cực nhỏ để không chết RAM Kaggle
    ACCUMULATION_STEPS = 4  
    EPOCHS = 1
    LR = 1e-4
    
    # Hàm Loss bắt buộc cho Multi-label
    criterion = nn.BCEWithLogitsLoss()
    
    # Optimizer SGD theo mô tả bài báo gốc
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=1e-4)
    
    # Scheduler: Cosine Annealing (Giảm dần learning rate hình sin)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    # Khởi tạo Scaler cho Mixed Precision
    scaler = GradScaler()
    
    print("Bắt đầu huấn luyện CHÍNH THỨC...")
    best_auroc = 0.0
    
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        
        train_loss = train_epoch(model, train_loader, criterion, optimizer, scaler, device, ACCUMULATION_STEPS)
        val_loss, val_auroc = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        
        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUROC: {val_auroc:.4f}")
        
        # Lưu mô hình (Checkpoint) nếu điểm AUROC cao hơn các epoch trước
        if val_auroc > best_auroc:
            best_auroc = val_auroc
            torch.save(model.state_dict(), 'best_vit_model.pth')
            print(f"--> [MỚI] Đã lưu mô hình tốt nhất với AUROC: {best_auroc:.4f}")
            
    print("HOÀN THÀNH HUẤN LUYỆN! File trọng số đã được lưu tại 'best_vit_model.pth'")

Đang chạy trên thiết bị: cuda
Đang nạp file dữ liệu CSV...
Đang khởi tạo mô hình Hybrid ViT...
Bắt đầu huấn luyện CHÍNH THỨC...

Epoch 1/1


Evaluating: 100%|██████████| 2793/2793 [06:37<00:00,  7.03it/s]


Train Loss: 0.3328 | Val Loss: 0.3198 | Val AUROC: 0.6806
--> [MỚI] Đã lưu mô hình tốt nhất với AUROC: 0.6806
HOÀN THÀNH HUẤN LUYỆN! File trọng số đã được lưu tại 'best_vit_model.pth'
